# Notebook 01 — ETL Pipeline — Student Version

## Goal

Build a **versioned, privacy-aware ML feature dataset** from the QBC12 Airbnb PostgreSQL database.

Final output:

- one row per `listing_id`
- one fixed `cutoff_date`
- features built only from data available before/on the cutoff
- target built from future calendar availability
- no raw PII columns in the final ML dataset

The next notebook will use this output for MLflow experiments. If this ETL is messy, the ML notebook will be garbage.

## What you must submit from this notebook

By the end, your notebook must save these files under `data/features/`:

```text
listing_availability_features_<version>.csv
listing_availability_features_<version>.parquet
listing_availability_features_<version>_metadata.json
listing_availability_features_<version>_validation_report.json
pii_audit_<version>.csv
```

The notebook must also show:

1. database connection check,
2. table/column inspection,
3. PII audit,
4. cutoff-date logic,
5. feature construction,
6. label construction,
7. validation checks.

## 0. Imports

These libraries are enough for the ETL notebook.

Install missing packages with:

```bash
pip install pandas numpy sqlalchemy psycopg2-binary pyarrow
```

In [1]:
import os
import json
import re
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

## 1. Configuration

These values define the dataset version and the time windows.

- `PAST_WINDOW_DAYS`: how much history is used for features.
- `FUTURE_WINDOW_DAYS`: how much future data is used for the target.
- `HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD`: the rule for the positive class.

If you change any of these, change `DATASET_VERSION`.

In [2]:
# -----------------------------
# ETL Configuration
# -----------------------------
DATASET_VERSION = "v1_student"

ENTITY_COLUMN = "listing_id"

PAST_WINDOW_DAYS = 90
FUTURE_WINDOW_DAYS = 30
HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD = 0.30

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
FEATURE_DIR = DATA_DIR / "features"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURE_DIR:", FEATURE_DIR)

PROJECT_ROOT: /home/alireza/PycharmProjects/Quera-Homeworks/HW02-A
FEATURE_DIR: /home/alireza/PycharmProjects/Quera-Homeworks/HW02-A/data/features


## 2. Database connection

Use your assigned student database user.

The QBC12 database is:

host: 185.50.38.163

port: 32112

database: qbc12_airbnb

Important:

- Keep `sslmode=disable`.
- Do not commit real passwords to Git.

In [ ]:
# -----------------------------
# Database Connection
# -----------------------------

# Clear old environment variables that may point to the wrong database.
# for key in ["PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]:
#     os.environ.pop(key, None)

DB_HOST = os.getenv("PGHOST", "185.50.38.163")
DB_PORT = int(os.getenv("PGPORT", "32112"))
DB_NAME = os.getenv("PGDATABASE", "qbc12_airbnb")
DB_USER = os.getenv("PGUSER", "")
DB_PASSWORD = os.getenv("PGPASSWORD", "")


if DB_USER == "student_your_username" or DB_PASSWORD == "student_your_password":
    raise ValueError("Replace DB_USER and DB_PASSWORD with your assigned database credentials.")

print("Connecting to:")
print("HOST:", DB_HOST)
print("PORT:", DB_PORT)
print("DB:", DB_NAME)
print("USER:", DB_USER)

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    query={"sslmode": "disable"},
)

engine = create_engine(db_url)

def read_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    """Run a SQL query through SQLAlchemy and return a Pandas DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)

with engine.connect() as conn:
    connection_check = conn.execute(
        text("""
        SELECT
            current_database() AS database,
            current_user AS user_name,
            inet_server_addr() AS server_ip,
            inet_server_port() AS server_port,
            now() AS checked_at;
        """)
    ).mappings().first()

dict(connection_check)

## 3. Inspect the available data

Before writing ETL, inspect the database.

You should confirm:

- which tables exist,
- which columns exist,
- how many rows each table has,
- whether important fields are missing.

In [4]:
tables_df = read_sql("""
SELECT
    table_schema,
    table_name,
    table_type
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
""")

tables_df

,table_schema,table_name,table_type
0,core,calendar_day,BASE TABLE
1,core,host,BASE TABLE
2,core,listing,BASE TABLE
3,core,neighbourhood,BASE TABLE
4,core,review,BASE TABLE


In [5]:
columns_df = read_sql("""
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'core'
ORDER BY table_schema, table_name, ordinal_position;
""")

columns_df

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable
0,core,calendar_day,1,listing_id,bigint,NO
1,core,calendar_day,2,date,date,NO
2,core,calendar_day,3,available,boolean,YES
3,core,calendar_day,4,price,numeric,YES
4,core,calendar_day,5,adjusted_price,numeric,YES
5,core,calendar_day,6,minimum_nights,integer,YES
6,core,calendar_day,7,maximum_nights,integer,YES
7,core,host,1,host_id,bigint,NO
8,core,host,2,host_pseudo_id,text,NO
9,core,host,3,is_superhost,boolean,YES


In [6]:
row_counts_df = read_sql("""
SELECT 'core.calendar_day' AS table_name, COUNT(*) AS row_count FROM core.calendar_day
UNION ALL
SELECT 'core.host' AS table_name, COUNT(*) AS row_count FROM core.host
UNION ALL
SELECT 'core.listing' AS table_name, COUNT(*) AS row_count FROM core.listing
UNION ALL
SELECT 'core.neighbourhood' AS table_name, COUNT(*) AS row_count FROM core.neighbourhood
UNION ALL
SELECT 'core.review' AS table_name, COUNT(*) AS row_count FROM core.review
ORDER BY table_name;
""")

row_counts_df

,table_name,row_count
0,core.calendar_day,3825200
1,core.host,9201
2,core.listing,10480
3,core.neighbourhood,22
4,core.review,501084


## 4. Data quality audit

This step decides which columns are safe and useful.

You must check at least:

1. calendar date range,
2. review date range,
3. whether `calendar_day.price` and `adjusted_price` are usable,
4. whether recent review windows are meaningful.

Do not include columns that are all-null or nearly useless.

In [7]:
calendar_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE price IS NULL) AS null_price,
    COUNT(*) FILTER (WHERE adjusted_price IS NULL) AS null_adjusted_price,
    COUNT(*) FILTER (WHERE available IS NULL) AS null_available,
    MIN(date) AS min_calendar_date,
    MAX(date) AS max_calendar_date
FROM core.calendar_day;
""")

review_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE comment_len IS NULL) AS null_comment_len,
    MIN(review_date) AS min_review_date,
    MAX(review_date) AS max_review_date
FROM core.review;
""")

display(calendar_quality_df)
display(review_quality_df)

,n_rows,null_price,null_adjusted_price,null_available,min_calendar_date,max_calendar_date
0,3825200,3825200,3825200,0,2025-09-11,2026-09-10


,n_rows,null_comment_len,min_review_date,max_review_date
0,501084,0,2010-08-22,2025-09-11


In [8]:
# Inspect small samples.
# Keep LIMIT small. Do not pull full raw calendar/review tables into Pandas.

for table_name in ["listing", "host", "neighbourhood", "review", "calendar_day"]:
    print(f"\n===== core.{table_name} =====")
    display(read_sql(f"SELECT * FROM core.{table_name} LIMIT 10;"))


===== core.listing =====


,listing_id,host_id,neighbourhood_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms_text,listing_price,minimum_nights,maximum_nights,instant_bookable,license
0,27886,97647,2,Private room,Private room in houseboat,2,1.0,1.0,1.5 baths,132.0,3,356,False,0363 974D 4986 7411 88D8
1,28871,124245,2,Private room,Private room in rental unit,2,1.0,1.0,1 shared bath,89.0,2,730,False,0363 607B EA74 0BD8 2F6F
2,29051,124245,1,Private room,Private room in condo,2,1.0,1.0,1 shared bath,61.0,2,730,False,0363 607B EA74 0BD8 2F6F
3,44391,194779,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,730,False,0363 E76E F06A C1DD 172C
4,48373,220434,8,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,1125,False,0363 4A2B A6AD 0196 F684
5,49552,225987,2,Entire home/apt,Entire guest suite,3,2.0,2.0,1 bath,322.0,3,1125,False,0363 576A D827 5085 6B83
6,50263,230246,1,Entire home/apt,Entire condo,4,2.0,3.0,1.5 baths,457.0,2,14,False,0363 7F3D 0BAE 28C8 C7D2
7,50515,231864,14,Entire home/apt,Entire townhouse,5,3.0,3.0,1.5 baths,198.0,7,18,False,0363 5DDB E495 A6D5 CEC6
8,50523,231946,2,Entire home/apt,Entire condo,2,1.0,1.0,1 bath,162.0,2,365,False,0363 22DC 0E52 B70B 0FB8
9,53921,252245,10,Entire home/apt,Entire rental unit,3,1.0,NaN,1 bath,NaN,1,21,False,0363 B43C B1D4 2666 3739



===== core.host =====


,host_id,host_pseudo_id,is_superhost
0,27837566,12a252de05fbf2f7ba9f57fa3baa099acd17e2a9c7efc7...,False
1,12840373,d9f7e79668b99a5cb7963bfb2430d8b6b960a0ec4e82bb...,False
2,226859324,cc90d30412e286c0525c7914d031807c8c00e017fe1915...,True
3,20204265,755d79bf4be51df2d85792123200e6641db8589551965e...,True
4,47981094,b40c45156e81696746b6eb51ef86aeccff7c6d7cd5eac2...,False
5,443406859,cfe69dd56cffef559069b30216f8954b57e1de886b92a3...,False
6,2626085,43c4cdd7f1413364dd809c6c92c20baed35faa51f84e76...,False
7,74999205,1033d2369452b0969c1f63061af401f4581a92873b5659...,True
8,7969106,1b41831025e74b04a73fe88e99233d09f39660eac85bf6...,False
9,6127483,8cffc835ea5e24b102cf446e9d369edad9f3a2bf91bfbc...,True



===== core.neighbourhood =====


,neighbourhood_id,name
0,1,Centrum-Oost
1,2,Centrum-West
2,3,Oostelijk Havengebied - Indische Buurt
3,4,Westerpark
4,5,Slotervaart
5,6,Bijlmer-Centrum
6,7,Geuzenveld - Slotermeer
7,8,Buitenveldert - Zuidas
8,9,Noord-West
9,10,IJburg - Zeeburgereiland



===== core.review =====


,review_id,listing_id,review_date,reviewer_id,reviewer_pseudo_id,comment_len
0,531281017,18062995,2019-09-17,73925523,2408ff6b678c0bdc30857dc6631d8762b57159ef9741c3...,392
1,531788922,18062995,2019-09-18,82572265,fb2e9dd6e8225de8231d388c9b9cf92b9c88f7e0823389...,13
2,532695264,18062995,2019-09-20,100595030,2503cafbac72a2a0a211282e2ed6e97c60058769ab3cf8...,467
3,537292491,18062995,2019-09-28,282542584,2612adccbea98416289b033feb314e33a3947244104fcf...,78
4,540324583,18062995,2019-10-03,116115856,7f6c33a5339d594d253c1aa0c9d5a2ae714321907af14c...,220
5,540880279,18062995,2019-10-04,65938182,b280baa9d72baf439c77b54c6ee11a42e4a9403fccec35...,145
6,542086221,18062995,2019-10-06,155392872,964cb3a74d1ccaaadd95915b842090390eed7754785aa5...,270
7,543734173,18062995,2019-10-08,52878420,5f477eaea2d8cd299828be54f31613eabe9e17072ff0bc...,204
8,544447121,18062995,2019-10-10,120972473,1314f5367fdcd67e16c7fb0828dad3be64be054d8042c2...,92
9,547826338,18062995,2019-10-16,31002341,6eedc8878c30f4380cbc920fd5ed63ff122aeba8f3bdf6...,91



===== core.calendar_day =====


,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,857771032141263073,2026-06-20,False,None,None,2,45
1,857771032141263073,2026-06-21,False,None,None,2,45
2,857771032141263073,2026-06-22,False,None,None,2,45
3,857771032141263073,2026-06-23,False,None,None,2,45
4,857771032141263073,2026-06-24,False,None,None,2,45
5,857771032141263073,2026-06-25,False,None,None,2,45
6,857771032141263073,2026-06-26,False,None,None,2,45
7,857771032141263073,2026-06-27,False,None,None,2,45
8,857771032141263073,2026-06-28,False,None,None,2,45
9,857771032141263073,2026-06-29,False,None,None,2,45


## 5. Choose the cutoff date

The cutoff separates features from the label.

Rules:

- Historical features use dates `history_start_date <= date <= cutoff_date`.
- The label uses dates `cutoff_date < date <= label_end_date`.
- The cutoff must have enough past calendar data and enough future calendar data.

For this homework, use a 90-day feature window and a 30-day label window.

In [9]:
range_df = read_sql("""
SELECT
    (SELECT MIN(date) FROM core.calendar_day) AS calendar_min_date,
    (SELECT MAX(date) FROM core.calendar_day) AS calendar_max_date,
    (SELECT MIN(review_date) FROM core.review) AS review_min_date,
    (SELECT MAX(review_date) FROM core.review) AS review_max_date;
""")

range_df

,calendar_min_date,calendar_max_date,review_min_date,review_max_date
0,2025-09-11,2026-09-10,2010-08-22,2025-09-11


In [10]:
calendar_min_date = pd.to_datetime(range_df.loc[0, "calendar_min_date"]).date()
calendar_max_date = pd.to_datetime(range_df.loc[0, "calendar_max_date"]).date()
review_min_date = pd.to_datetime(range_df.loc[0, "review_min_date"]).date()
review_max_date = pd.to_datetime(range_df.loc[0, "review_max_date"]).date()

# Earliest cutoff must leave enough historical calendar days.
earliest_cutoff_allowed_by_calendar = calendar_min_date + timedelta(days=PAST_WINDOW_DAYS)

# Latest cutoff must leave enough future calendar days for the label.
latest_cutoff_allowed_by_calendar = calendar_max_date - timedelta(days=FUTURE_WINDOW_DAYS)

assert earliest_cutoff_allowed_by_calendar <= latest_cutoff_allowed_by_calendar, (
    "Calendar table does not contain enough dates for the requested past/future windows."
)

# Use the most recent cutoff that still has a complete 30-day future label window.
# This maximizes the amount of historical calendar data available while avoiding label leakage.
cutoff_date = latest_cutoff_allowed_by_calendar

history_start_date = cutoff_date - timedelta(days=PAST_WINDOW_DAYS)
label_end_date = cutoff_date + timedelta(days=FUTURE_WINDOW_DAYS)

assert history_start_date >= calendar_min_date
assert label_end_date <= calendar_max_date
assert history_start_date <= cutoff_date < label_end_date

print("calendar_min_date:", calendar_min_date)
print("calendar_max_date:", calendar_max_date)
print("review_min_date:", review_min_date)
print("review_max_date:", review_max_date)
print("earliest_cutoff_allowed_by_calendar:", earliest_cutoff_allowed_by_calendar)
print("latest_cutoff_allowed_by_calendar:", latest_cutoff_allowed_by_calendar)
print("chosen cutoff_date:", cutoff_date)
print("history_start_date:", history_start_date)
print("label_end_date:", label_end_date)

calendar_min_date: 2025-09-11
calendar_max_date: 2026-09-10
review_min_date: 2010-08-22
review_max_date: 2025-09-11
earliest_cutoff_allowed_by_calendar: 2025-12-10
latest_cutoff_allowed_by_calendar: 2026-08-11
chosen cutoff_date: 2026-08-11
history_start_date: 2026-05-13
label_end_date: 2026-09-10


## 6. PII audit

Raw identifiers can be needed for joins, but they must not become model features.

Your final ML feature table must not contain:

- `host_id`
- `host_pseudo_id`
- `review_id`
- `reviewer_id`
- `reviewer_pseudo_id`
- `license`
- raw text fields that may contain sensitive information

`listing_id` may stay as an entity key, but it must be excluded from model inputs later.

In [11]:
# PII audit table.
# Raw identifiers are allowed for joins/auditing, but they must not become model input features.

pii_audit = pd.DataFrame([
    {
        "table": "listing",
        "column": "listing_id",
        "pii_type": "entity identifier",
        "decision": "keep as entity key only",
        "reason": "needed to define one row per listing; exclude from model inputs"
    },
    {
        "table": "listing",
        "column": "host_id",
        "pii_type": "person/account identifier",
        "decision": "use for joins only, drop before final ML table",
        "reason": "links a listing to a host identity"
    },
    {
        "table": "listing",
        "column": "license",
        "pii_type": "regulatory/property identifier",
        "decision": "drop",
        "reason": "can identify a licensed property or host; not needed for modeling"
    },
    {
        "table": "host",
        "column": "host_id",
        "pii_type": "person/account identifier",
        "decision": "use for joins only, drop before final ML table",
        "reason": "needed only to join host attributes to listings"
    },
    {
        "table": "host",
        "column": "host_pseudo_id",
        "pii_type": "pseudonymous person/account identifier",
        "decision": "drop",
        "reason": "identity-linking pseudonymous identifier; not useful as an ML feature"
    },
    {
        "table": "review",
        "column": "review_id",
        "pii_type": "review row identifier",
        "decision": "aggregate only, do not keep raw",
        "reason": "raw review IDs are not useful features and can link back to source rows"
    },
    {
        "table": "review",
        "column": "reviewer_id",
        "pii_type": "person/account identifier",
        "decision": "aggregate only, do not keep raw",
        "reason": "used only for reviewer counts; individual reviewer identities are not kept"
    },
    {
        "table": "review",
        "column": "reviewer_pseudo_id",
        "pii_type": "pseudonymous person/account identifier",
        "decision": "drop",
        "reason": "identity-linking pseudonymous identifier; not needed after aggregation"
    },
    {
        "table": "review",
        "column": "comment_len",
        "pii_type": "safe derived text length",
        "decision": "aggregate only",
        "reason": "keeps only numeric review-length statistics, not raw review text"
    },
])

pii_audit

,table,column,pii_type,decision,reason
0,listing,listing_id,entity identifier,keep as entity key only,needed to define one row per listing; exclude ...
1,listing,host_id,person/account identifier,"use for joins only, drop before final ML table",links a listing to a host identity
2,listing,license,regulatory/property identifier,drop,can identify a licensed property or host; not ...
3,host,host_id,person/account identifier,"use for joins only, drop before final ML table",needed only to join host attributes to listings
4,host,host_pseudo_id,pseudonymous person/account identifier,drop,identity-linking pseudonymous identifier; not ...
5,review,review_id,review row identifier,"aggregate only, do not keep raw",raw review IDs are not useful features and can...
6,review,reviewer_id,person/account identifier,"aggregate only, do not keep raw",used only for reviewer counts; individual revi...
7,review,reviewer_pseudo_id,pseudonymous person/account identifier,drop,identity-linking pseudonymous identifier; not ...
8,review,comment_len,safe derived text length,aggregate only,"keeps only numeric review-length statistics, n..."


## 7. Extract static tables

`listing`, `host`, and `neighbourhood` are small enough to load directly.

Do not load full `review` or `calendar_day` into Pandas. Those must be aggregated in SQL later.

In [12]:
# Load only the static columns needed for joins and feature building.
#
# Important performance fix:
# The remote DB has a strict statement_timeout. A plain
#     SELECT ... FROM core.listing
# can be canceled even though listing is much smaller than calendar_day.
#
# This cell therefore reads listing with keyset pagination on listing_id.
# It also loads only the host/neighbourhood rows referenced by those listings.
# No raw PII columns such as license or host_pseudo_id are selected.

from collections.abc import Iterable


def existing_columns(table_name: str) -> set[str]:
    return set(
        columns_df.loc[
            (columns_df["table_schema"] == "core") & (columns_df["table_name"] == table_name),
            "column_name"
        ].tolist()
    )


def require_columns(table_name: str, required: list[str]) -> None:
    cols = existing_columns(table_name)
    missing = sorted(set(required) - cols)
    if missing:
        raise ValueError(f"core.{table_name} is missing required columns: {missing}")


def add_select(select_parts: list[str], table_cols: set[str], column_name: str, expression: str | None = None) -> None:
    if column_name in table_cols:
        select_parts.append(expression or column_name)


def _python_scalar(value):
    """Convert numpy scalar IDs to plain Python values for SQL parameters."""
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value


def read_table_by_keyset_chunks(
    *,
    table_name: str,
    id_column: str,
    select_parts: list[str],
    batch_size: int = 500,
    max_batches: int | None = None,
) -> pd.DataFrame:
    """
    Read a table in small keyset-pagination chunks.

    This avoids long-running full-table SELECT statements that get canceled by
    PostgreSQL statement_timeout. It assumes id_column is sortable and indexed
    or at least reasonably selective.
    """
    frames: list[pd.DataFrame] = []
    last_id = None
    batches_read = 0
    select_sql = ",\n        ".join(select_parts)

    while True:
        where_sql = "" if last_id is None else f"WHERE {id_column} > :last_id"
        params = {"batch_size": batch_size}
        if last_id is not None:
            params["last_id"] = last_id

        chunk = read_sql(
            f"""
            SELECT
                {select_sql}
            FROM core.{table_name}
            {where_sql}
            ORDER BY {id_column}
            LIMIT :batch_size;
            """,
            params=params,
        )

        if chunk.empty:
            break

        frames.append(chunk)
        batches_read += 1
        last_id = _python_scalar(chunk[id_column].iloc[-1])
        print(f"core.{table_name}: loaded batch {batches_read}, rows so far = {sum(len(x) for x in frames):,}")

        if len(chunk) < batch_size:
            break
        if max_batches is not None and batches_read >= max_batches:
            print(f"Stopped early after max_batches={max_batches}. Remove max_batches for the full dataset.")
            break

    if not frames:
        return pd.DataFrame(columns=[id_column])

    result = pd.concat(frames, ignore_index=True)
    result = result.drop_duplicates(subset=[id_column]).reset_index(drop=True)
    return result


def read_table_for_ids_in_chunks(
    *,
    table_name: str,
    id_column: str,
    ids: Iterable,
    select_parts: list[str],
    batch_size: int = 500,
) -> pd.DataFrame:
    """
    Read only rows whose IDs are needed for joins.

    This avoids scanning all host/neighbourhood rows when the final entity is listing.
    """
    clean_ids = [_python_scalar(x) for x in pd.Series(list(ids)).dropna().unique().tolist()]
    if not clean_ids:
        return pd.DataFrame(columns=[id_column])

    frames: list[pd.DataFrame] = []
    select_sql = ",\n        ".join(select_parts)

    for i in range(0, len(clean_ids), batch_size):
        id_batch = clean_ids[i:i + batch_size]
        print(f"core.{table_name}: loading referenced IDs {i + 1:,}-{i + len(id_batch):,} of {len(clean_ids):,}")
        chunk = read_sql(
            f"""
            SELECT
                {select_sql}
            FROM core.{table_name}
            WHERE {id_column} = ANY(:ids);
            """,
            params={"ids": id_batch},
        )
        if not chunk.empty:
            frames.append(chunk)

    if not frames:
        return pd.DataFrame(columns=[id_column])

    result = pd.concat(frames, ignore_index=True)
    result = result.drop_duplicates(subset=[id_column]).reset_index(drop=True)
    return result


# -----------------------------
# listing
# -----------------------------
listing_cols = existing_columns("listing")
require_columns("listing", ["listing_id", "host_id", "neighbourhood_id"])

listing_select_parts = []
for col in [
    "listing_id",
    "host_id",
    "neighbourhood_id",
    "room_type",
    "property_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms_text",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable",
]:
    add_select(listing_select_parts, listing_cols, col)

# Compatibility fallback in case another copy of the DB uses price instead of listing_price.
if "listing_price" not in listing_cols and "price" in listing_cols:
    listing_select_parts.append(
        "NULLIF(REGEXP_REPLACE(price::text, '[^0-9.]', '', 'g'), '')::numeric AS listing_price"
    )

# Start conservatively. If this still times out, reduce batch_size to 100 or 50.
listing_df = read_table_by_keyset_chunks(
    table_name="listing",
    id_column="listing_id",
    select_parts=listing_select_parts,
    batch_size=500,
)

# -----------------------------
# host
# -----------------------------
host_cols = existing_columns("host")
require_columns("host", ["host_id"])

host_select_parts = []
for col in [
    "host_id",
    "is_superhost",
    "host_since",
    "host_listings_count",
    "host_total_listings_count",
    "host_response_rate",
    "host_acceptance_rate",
    "host_identity_verified",
]:
    add_select(host_select_parts, host_cols, col)

# Only load hosts that appear in listing_df. Do not select host_pseudo_id.
host_df = read_table_for_ids_in_chunks(
    table_name="host",
    id_column="host_id",
    ids=listing_df["host_id"] if "host_id" in listing_df.columns else [],
    select_parts=host_select_parts,
    batch_size=500,
)

# -----------------------------
# neighbourhood
# -----------------------------
neighbourhood_cols = existing_columns("neighbourhood")
require_columns("neighbourhood", ["neighbourhood_id"])

neighbourhood_select_parts = ["neighbourhood_id"]

if "name" in neighbourhood_cols:
    neighbourhood_select_parts.append("name AS neighbourhood_name")
elif "neighbourhood_name" in neighbourhood_cols:
    neighbourhood_select_parts.append("neighbourhood_name")
elif "neighbourhood" in neighbourhood_cols:
    neighbourhood_select_parts.append("neighbourhood AS neighbourhood_name")
else:
    raise ValueError("Could not find a neighbourhood name column in core.neighbourhood.")

if "neighbourhood_group" in neighbourhood_cols:
    neighbourhood_select_parts.append("neighbourhood_group")

# Only load neighbourhoods referenced by listing_df.
neighbourhood_df = read_table_for_ids_in_chunks(
    table_name="neighbourhood",
    id_column="neighbourhood_id",
    ids=listing_df["neighbourhood_id"] if "neighbourhood_id" in listing_df.columns else [],
    select_parts=neighbourhood_select_parts,
    batch_size=500,
)

print("listing:", listing_df.shape)
print("host:", host_df.shape)
print("neighbourhood:", neighbourhood_df.shape)

display(listing_df.head())
display(host_df.head())
display(neighbourhood_df.head())



core.listing: loaded batch 1, rows so far = 500
core.listing: loaded batch 2, rows so far = 1,000
core.listing: loaded batch 3, rows so far = 1,500
core.listing: loaded batch 4, rows so far = 2,000
core.listing: loaded batch 5, rows so far = 2,500
core.listing: loaded batch 6, rows so far = 3,000
core.listing: loaded batch 7, rows so far = 3,500
core.listing: loaded batch 8, rows so far = 4,000
core.listing: loaded batch 9, rows so far = 4,500
core.listing: loaded batch 10, rows so far = 5,000
core.listing: loaded batch 11, rows so far = 5,500
core.listing: loaded batch 12, rows so far = 6,000
core.listing: loaded batch 13, rows so far = 6,500
core.listing: loaded batch 14, rows so far = 7,000
core.listing: loaded batch 15, rows so far = 7,500
core.listing: loaded batch 16, rows so far = 8,000
core.listing: loaded batch 17, rows so far = 8,500
core.listing: loaded batch 18, rows so far = 9,000
core.listing: loaded batch 19, rows so far = 9,500
core.listing: loaded batch 20, rows so far

,listing_id,host_id,neighbourhood_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms_text,listing_price,minimum_nights,maximum_nights,instant_bookable
0,27886,97647,2,Private room,Private room in houseboat,2,1.0,1.0,1.5 baths,132.0,3,356,False
1,28871,124245,2,Private room,Private room in rental unit,2,1.0,1.0,1 shared bath,89.0,2,730,False
2,29051,124245,1,Private room,Private room in condo,2,1.0,1.0,1 shared bath,61.0,2,730,False
3,44391,194779,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,730,False
4,48373,220434,8,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,1125,False


,host_id,is_superhost
0,6127483,True
1,5102632,True
2,3509817,False
3,12448020,True
4,14122005,True


,neighbourhood_id,neighbourhood_name
0,1,Centrum-Oost
1,2,Centrum-West
2,3,Oostelijk Havengebied - Indische Buurt
3,4,Westerpark
4,5,Slotervaart


## 8. Clean static fields

Convert database values into ML-friendly columns.

Required work:

- convert booleans to boolean dtype,
- convert numeric listing columns to numeric dtype,
- parse `bathrooms_text` into a numeric `bathrooms` feature.

In [13]:
# Convert database values into ML-friendly dtypes.

def to_nullable_bool(value):
    """Normalize common true/false encodings to pandas nullable boolean values."""
    if pd.isna(value):
        return pd.NA

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    value_str = str(value).strip().lower()

    if value_str in {"t", "true", "1", "yes", "y"}:
        return True
    if value_str in {"f", "false", "0", "no", "n"}:
        return False

    return pd.NA

def clean_numeric_series(series: pd.Series) -> pd.Series:
    """Convert numeric-looking values, including '$1,234' strings, to numbers."""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    cleaned = (
        series.astype("string")
        .str.replace(r"[^0-9.\-]", "", regex=True)
        .replace("", pd.NA)
    )

    return pd.to_numeric(cleaned, errors="coerce")

def clean_percentage_series(series: pd.Series) -> pd.Series:
    """Convert values like '95%' to 0.95. Numeric values > 1 are assumed to be percentages."""
    numeric = clean_numeric_series(series)
    return pd.Series(
        np.where(numeric > 1, numeric / 100.0, numeric),
        index=series.index,
        dtype="float64"
    )

def parse_bathrooms_text(value):
    """
    Convert examples such as:
    - '1 bath' -> 1.0
    - '1.5 baths' -> 1.5
    - '1 shared bath' -> 1.0
    - 'Half-bath' -> 0.5
    """
    if pd.isna(value):
        return np.nan

    text_value = str(value).strip().lower()

    if "half" in text_value:
        return 0.5

    match = re.search(r"(\d+(?:\.\d+)?)", text_value)
    if match:
        return float(match.group(1))

    return np.nan

# Normalize boolean columns.
for df, col in [
    (listing_df, "instant_bookable"),
    (host_df, "is_superhost"),
    (host_df, "host_identity_verified"),
]:
    if col in df.columns:
        df[col] = df[col].apply(to_nullable_bool).astype("boolean")

# Normalize numeric listing columns.
numeric_cols_listing = [
    "accommodates",
    "bedrooms",
    "beds",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
]

for col in numeric_cols_listing:
    if col in listing_df.columns:
        listing_df[col] = clean_numeric_series(listing_df[col])

# Parse bathrooms_text into a numeric bathrooms feature, then drop raw text later.
if "bathrooms_text" in listing_df.columns:
    listing_df["bathrooms"] = listing_df["bathrooms_text"].apply(parse_bathrooms_text).astype("float64")

# Normalize optional host numeric/percentage columns when present.
for col in ["host_listings_count", "host_total_listings_count"]:
    if col in host_df.columns:
        host_df[col] = clean_numeric_series(host_df[col])

for col in ["host_response_rate", "host_acceptance_rate"]:
    if col in host_df.columns:
        host_df[col] = clean_percentage_series(host_df[col])

# Optional host age feature, if the DB copy has host_since.
if "host_since" in host_df.columns:
    host_df["host_since"] = pd.to_datetime(host_df["host_since"], errors="coerce")
    host_df["host_years_active"] = (
        (pd.to_datetime(cutoff_date) - host_df["host_since"]).dt.days / 365.25
    ).clip(lower=0)

print("listing dtypes:")
display(listing_df.dtypes)

print("host dtypes:")
display(host_df.dtypes)

display(listing_df.head())
display(host_df.head())

listing dtypes:


listing_id            int64
host_id               int64
neighbourhood_id      int64
room_type               str
property_type           str
accommodates          int64
bedrooms            float64
beds                float64
bathrooms_text          str
listing_price       float64
minimum_nights        int64
maximum_nights        int64
instant_bookable    boolean
bathrooms           float64
dtype: object

host dtypes:


host_id           int64
is_superhost    boolean
dtype: object

,listing_id,host_id,neighbourhood_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms_text,listing_price,minimum_nights,maximum_nights,instant_bookable,bathrooms
0,27886,97647,2,Private room,Private room in houseboat,2,1.0,1.0,1.5 baths,132.0,3,356,False,1.5
1,28871,124245,2,Private room,Private room in rental unit,2,1.0,1.0,1 shared bath,89.0,2,730,False,1.0
2,29051,124245,1,Private room,Private room in condo,2,1.0,1.0,1 shared bath,61.0,2,730,False,1.0
3,44391,194779,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,730,False,1.5
4,48373,220434,8,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,1125,False,1.5


,host_id,is_superhost
0,6127483,True
1,5102632,True
2,3509817,False
3,12448020,True
4,14122005,True


## 9. Build static listing features

Join:

- `listing` → `host`
- `listing` → `neighbourhood`
- host-level aggregate `host_listing_count`

Final static features should be one row per `listing_id`.

Do not keep raw `host_id`, `host_pseudo_id`, `neighbourhood_id`, `license`, or `bathrooms_text` in the final static feature table.

In [14]:
# Host-level aggregate: how many listings each host has in this database.
host_listing_features = (
    listing_df
    .groupby("host_id", as_index=False)
    .agg(host_listing_count=("listing_id", "nunique"))
)

# Merge static tables.
base_listing_features = (
    listing_df
    .merge(host_df, on="host_id", how="left", suffixes=("", "_host"))
    .merge(host_listing_features, on="host_id", how="left")
    .merge(neighbourhood_df, on="neighbourhood_id", how="left")
)

# Privacy-safe static feature columns.
# Keep listing_id only as the entity key. Do not keep host_id, host_pseudo_id, license, or bathrooms_text.
candidate_static_feature_cols = [
    "listing_id",
    "room_type",
    "property_type",
    "neighbourhood_name",
    "neighbourhood_group",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable",
    "is_superhost",
    "host_years_active",
    "host_listings_count",
    "host_total_listings_count",
    "host_response_rate",
    "host_acceptance_rate",
    "host_identity_verified",
    "host_listing_count",
]

static_feature_cols = [
    col for col in candidate_static_feature_cols
    if col in base_listing_features.columns
]

static_features = base_listing_features[static_feature_cols].copy()

assert "listing_id" in static_features.columns
assert static_features["listing_id"].duplicated().sum() == 0

print(static_features.shape)
display(static_features.head())

(10480, 14)


,listing_id,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,instant_bookable,is_superhost,host_listing_count
0,27886,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,132.0,3,356,False,True,1
1,28871,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,89.0,2,730,False,True,2
2,29051,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,61.0,2,730,False,True,2
3,44391,Entire home/apt,Entire rental unit,Centrum-Oost,4,2.0,NaN,1.5,NaN,3,730,False,False,1
4,48373,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,4,2.0,NaN,1.5,NaN,3,1125,False,False,1


## 10. Build review features in SQL

Do not load raw `core.review` into Pandas.

Build one row per listing in SQL.

Required output columns:

- `listing_id`
- `total_reviews_before_cutoff`
- `unique_reviewers_before_cutoff`
- `avg_comment_len_before_cutoff`
- `max_comment_len_before_cutoff`
- `days_since_last_review`

Use only reviews where `review_date <= cutoff_date`.

In [15]:
# Build one row per listing from reviews, using only reviews on/before cutoff_date.
# Do not bring raw review rows or raw review identifiers into the final ML table.

review_cols = existing_columns("review")
require_columns("review", ["listing_id", "review_date"])

reviewer_distinct_col = None
for candidate in ["reviewer_id", "reviewer_pseudo_id"]:
    if candidate in review_cols:
        reviewer_distinct_col = candidate
        break

if "comment_len" in review_cols:
    comment_len_expr = "comment_len::numeric"
else:
    # Compatibility fallback for DB copies that expose raw review text.
    comment_text_col = None
    for candidate in ["comments", "comment", "review_comment", "review_text"]:
        if candidate in review_cols:
            comment_text_col = candidate
            break

    comment_len_expr = (
        f"LENGTH(COALESCE({comment_text_col}::text, ''))::numeric"
        if comment_text_col
        else "NULL::numeric"
    )

unique_reviewers_expr = (
    f"COUNT(DISTINCT {reviewer_distinct_col})::integer"
    if reviewer_distinct_col
    else "NULL::integer"
)

review_features = read_sql(
    f"""
    SELECT
        listing_id,
        COUNT(*)::integer AS total_reviews_before_cutoff,
        {unique_reviewers_expr} AS unique_reviewers_before_cutoff,
        AVG({comment_len_expr}) AS avg_comment_len_before_cutoff,
        MAX({comment_len_expr}) AS max_comment_len_before_cutoff,
        (CAST(:cutoff_date AS date) - MAX(review_date)::date)::integer AS days_since_last_review
    FROM core.review
    WHERE review_date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={"cutoff_date": cutoff_date},
)

for col in [
    "total_reviews_before_cutoff",
    "unique_reviewers_before_cutoff",
    "avg_comment_len_before_cutoff",
    "max_comment_len_before_cutoff",
    "days_since_last_review",
]:
    if col in review_features.columns:
        review_features[col] = pd.to_numeric(review_features[col], errors="coerce")

assert review_features["listing_id"].duplicated().sum() == 0

print(review_features.shape)
display(review_features.head())

(9383, 6)


,listing_id,total_reviews_before_cutoff,unique_reviewers_before_cutoff,avg_comment_len_before_cutoff,max_comment_len_before_cutoff,days_since_last_review
0,27886,311,311,302.167203,1917.0,338
1,28871,732,729,201.236339,1265.0,338
2,29051,849,841,245.108363,2253.0,337
3,44391,42,42,242.309524,891.0,1452
4,48373,5,5,272.200000,949.0,835


## 11. Build calendar history features in SQL

Do not load raw `core.calendar_day` into Pandas.

Build historical availability features using:

- 90-day history window,
- 30-day recent history window.

The remote database may cancel a single large calendar aggregation because of `statement_timeout`, so this notebook aggregates calendar data in small date chunks and combines only listing-level aggregate rows in Pandas.

Do not include calendar price features unless your audit proves they are usable.


In [16]:
# Historical calendar features.
#
# The direct 90-day GROUP BY query may be canceled by the remote PostgreSQL
# statement_timeout because core.calendar_day is the largest table.
#
# To stay ETL-safe and avoid loading raw calendar rows into Pandas, we query
# small date chunks, aggregate each chunk in SQL, and combine only the grouped
# listing-level results in Pandas.

from datetime import timedelta

calendar_cols = existing_columns("calendar_day")
require_columns("calendar_day", ["listing_id", "date", "available"])

available_flag_sql = """
CASE
    WHEN LOWER(available::text) IN ('t', 'true', '1', 'yes', 'y', 'available') THEN 1
    WHEN LOWER(available::text) IN ('f', 'false', '0', 'no', 'n', 'unavailable') THEN 0
    ELSE NULL
END
"""

minimum_nights_sum_sql = (
    "COALESCE(SUM(minimum_nights::numeric), 0)::numeric"
    if "minimum_nights" in calendar_cols
    else "0::numeric"
)
minimum_nights_count_sql = (
    "COUNT(minimum_nights)::integer"
    if "minimum_nights" in calendar_cols
    else "0::integer"
)
maximum_nights_sum_sql = (
    "COALESCE(SUM(maximum_nights::numeric), 0)::numeric"
    if "maximum_nights" in calendar_cols
    else "0::numeric"
)
maximum_nights_count_sql = (
    "COUNT(maximum_nights)::integer"
    if "maximum_nights" in calendar_cols
    else "0::integer"
)


def _calendar_window_batches(start_inclusive, end_exclusive, chunk_days: int = 7):
    """Yield half-open date batches: [batch_start, batch_end)."""
    batch_start = start_inclusive
    while batch_start < end_exclusive:
        batch_end = min(batch_start + timedelta(days=chunk_days), end_exclusive)
        yield batch_start, batch_end
        batch_start = batch_end


def read_calendar_window_aggregates(
    start_inclusive,
    end_exclusive,
    *,
    window_days: int,
    chunk_days: int = 7,
    name: str = "history",
) -> pd.DataFrame:
    """
    Aggregate core.calendar_day for a date window without pulling raw rows.

    The SQL returns one row per listing per small date chunk. Pandas then combines
    those already-aggregated rows into one row per listing.
    """
    batch_frames = []

    for batch_start, batch_end in _calendar_window_batches(start_inclusive, end_exclusive, chunk_days):
        print(f"{name}: reading {batch_start} <= date < {batch_end}")

        batch_df = read_sql(
            f"""
            SELECT
                listing_id,
                COUNT(*)::integer AS observed_days,
                COALESCE(SUM({available_flag_sql}), 0)::numeric AS available_sum,
                COUNT({available_flag_sql})::integer AS availability_non_null_days,
                {minimum_nights_sum_sql} AS minimum_nights_sum,
                {minimum_nights_count_sql} AS minimum_nights_non_null_days,
                {maximum_nights_sum_sql} AS maximum_nights_sum,
                {maximum_nights_count_sql} AS maximum_nights_non_null_days
            FROM core.calendar_day
            WHERE date >= CAST(:batch_start AS date)
              AND date < CAST(:batch_end AS date)
            GROUP BY listing_id;
            """,
            params={"batch_start": batch_start, "batch_end": batch_end},
        )

        if not batch_df.empty:
            batch_frames.append(batch_df)

    if not batch_frames:
        return pd.DataFrame(
            columns=[
                "listing_id",
                f"calendar_days_observed_last_{window_days}d",
                f"available_days_last_{window_days}d",
                f"available_rate_last_{window_days}d",
                f"avg_minimum_nights_calendar_last_{window_days}d",
                f"avg_maximum_nights_calendar_last_{window_days}d",
            ]
        )

    raw_aggregates = pd.concat(batch_frames, ignore_index=True)

    numeric_cols = [
        "observed_days",
        "available_sum",
        "availability_non_null_days",
        "minimum_nights_sum",
        "minimum_nights_non_null_days",
        "maximum_nights_sum",
        "maximum_nights_non_null_days",
    ]

    for col in numeric_cols:
        raw_aggregates[col] = pd.to_numeric(raw_aggregates[col], errors="coerce").fillna(0)

    grouped = raw_aggregates.groupby("listing_id", as_index=False)[numeric_cols].sum()

    result = grouped[["listing_id"]].copy()
    result[f"calendar_days_observed_last_{window_days}d"] = grouped["observed_days"].astype(int)
    result[f"available_days_last_{window_days}d"] = grouped["available_sum"].astype(int)
    result[f"available_rate_last_{window_days}d"] = (
        grouped["available_sum"] / grouped["availability_non_null_days"].replace(0, np.nan)
    )
    result[f"avg_minimum_nights_calendar_last_{window_days}d"] = (
        grouped["minimum_nights_sum"] / grouped["minimum_nights_non_null_days"].replace(0, np.nan)
    )
    result[f"avg_maximum_nights_calendar_last_{window_days}d"] = (
        grouped["maximum_nights_sum"] / grouped["maximum_nights_non_null_days"].replace(0, np.nan)
    )

    return result


# Original SQL logic was:
# date > cutoff_date - N days AND date <= cutoff_date
# For daily dates, the equivalent half-open Python range is:
# cutoff_date - N days + 1 <= date < cutoff_date + 1 day
history_90d_start = cutoff_date - timedelta(days=PAST_WINDOW_DAYS) + timedelta(days=1)
history_30d_start = cutoff_date - timedelta(days=30) + timedelta(days=1)
history_end_exclusive = cutoff_date + timedelta(days=1)

calendar_features_90d = read_calendar_window_aggregates(
    history_90d_start,
    history_end_exclusive,
    window_days=90,
    chunk_days=7,
    name="history_90d",
)

calendar_features_30d = read_calendar_window_aggregates(
    history_30d_start,
    history_end_exclusive,
    window_days=30,
    chunk_days=7,
    name="history_30d",
)

calendar_features_all = calendar_features_90d.merge(
    calendar_features_30d,
    on="listing_id",
    how="outer",
)

for col in calendar_features_all.columns:
    if col != "listing_id":
        calendar_features_all[col] = pd.to_numeric(calendar_features_all[col], errors="coerce")

assert calendar_features_all["listing_id"].duplicated().sum() == 0

print(calendar_features_all.shape)
display(calendar_features_all.head())


history_90d: reading 2026-05-14 <= date < 2026-05-21
history_90d: reading 2026-05-21 <= date < 2026-05-28
history_90d: reading 2026-05-28 <= date < 2026-06-04
history_90d: reading 2026-06-04 <= date < 2026-06-11
history_90d: reading 2026-06-11 <= date < 2026-06-18
history_90d: reading 2026-06-18 <= date < 2026-06-25
history_90d: reading 2026-06-25 <= date < 2026-07-02
history_90d: reading 2026-07-02 <= date < 2026-07-09
history_90d: reading 2026-07-09 <= date < 2026-07-16
history_90d: reading 2026-07-16 <= date < 2026-07-23
history_90d: reading 2026-07-23 <= date < 2026-07-30
history_90d: reading 2026-07-30 <= date < 2026-08-06
history_90d: reading 2026-08-06 <= date < 2026-08-12
history_30d: reading 2026-07-13 <= date < 2026-07-20
history_30d: reading 2026-07-20 <= date < 2026-07-27
history_30d: reading 2026-07-27 <= date < 2026-08-03
history_30d: reading 2026-08-03 <= date < 2026-08-10
history_30d: reading 2026-08-10 <= date < 2026-08-12
(10480, 11)


,listing_id,calendar_days_observed_last_90d,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,calendar_days_observed_last_30d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d
0,27886,90,0,0.000000,3.0,30.0,30,0,0.000000,3.0,30.0
1,28871,90,45,0.500000,2.0,730.0,30,14,0.466667,2.0,730.0
2,29051,90,42,0.466667,2.0,730.0,30,16,0.533333,2.0,730.0
3,44391,90,0,0.000000,3.0,730.0,30,0,0.000000,3.0,730.0
4,48373,90,0,0.000000,3.0,1125.0,30,0,0.000000,3.0,1125.0


## 12. Build the target label

The label is built from future calendar availability.

Positive class:

```text
high_demand_proxy = 1 if future_available_rate_30d <= 0.30
```

This is not confirmed booking demand. It is a low-availability proxy.

In [17]:
# Build the future low-availability proxy label.
# Label window: cutoff_date < date <= label_end_date
#
# Use the same chunked SQL aggregation pattern as the history features so this
# cell also survives strict PostgreSQL statement_timeout settings.

future_start_inclusive = cutoff_date + timedelta(days=1)
future_end_exclusive = label_end_date + timedelta(days=1)

future_calendar_30d = read_calendar_window_aggregates(
    future_start_inclusive,
    future_end_exclusive,
    window_days=30,
    chunk_days=7,
    name="future_30d",
)

label_df = future_calendar_30d.rename(
    columns={
        "calendar_days_observed_last_30d": "future_calendar_days_observed_30d",
        "available_days_last_30d": "future_available_days_30d",
        "available_rate_last_30d": "future_available_rate_30d",
    }
)

# These are future-only helper columns from the generic calendar aggregation;
# they are not needed as labels and should not enter the final dataset.
label_df = label_df.drop(
    columns=[
        "avg_minimum_nights_calendar_last_30d",
        "avg_maximum_nights_calendar_last_30d",
    ],
    errors="ignore",
)

label_df["high_demand_proxy"] = (
    label_df["future_available_rate_30d"] <= HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD
).astype(int)

for col in [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]:
    label_df[col] = pd.to_numeric(label_df[col], errors="coerce")

label_df["high_demand_proxy"] = label_df["high_demand_proxy"].fillna(0).astype("Int64")

assert label_df["listing_id"].duplicated().sum() == 0
assert label_df["high_demand_proxy"].isna().sum() == 0

print(label_df.shape)
display(label_df.head())


future_30d: reading 2026-08-12 <= date < 2026-08-19
future_30d: reading 2026-08-19 <= date < 2026-08-26
future_30d: reading 2026-08-26 <= date < 2026-09-02
future_30d: reading 2026-09-02 <= date < 2026-09-09
future_30d: reading 2026-09-09 <= date < 2026-09-11
(10480, 5)


,listing_id,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy
0,27886,30,0,0.0,1
1,28871,30,21,0.7,0
2,29051,30,0,0.0,1
3,44391,30,0,0.0,1
4,48373,30,0,0.0,1


In [18]:
# Check label balance.

label_distribution = (
    label_df["high_demand_proxy"]
    .value_counts(dropna=False)
    .rename_axis("high_demand_proxy")
    .reset_index(name="count")
)

label_distribution["percentage"] = (
    label_distribution["count"] / label_distribution["count"].sum()
).round(4)

label_distribution

,high_demand_proxy,count,percentage
0,1,7994,0.7628
1,0,2486,0.2372


## 13. Join feature groups and label

Join all feature groups into one ML-ready table.

The final granularity must be:

```text
one row = one listing_id at one cutoff_date
```

Use an inner join with `label_df`, because rows without a target cannot be used for supervised learning.

In [19]:
# Join static, review, calendar history, and label features.
# Use an inner join with label_df so every row has a supervised-learning target.

feature_df = (
    static_features
    .merge(review_features, on="listing_id", how="left")
    .merge(calendar_features_all, on="listing_id", how="left")
    .merge(label_df, on="listing_id", how="inner")
)

feature_df["cutoff_date"] = pd.to_datetime(cutoff_date).date().isoformat()
feature_df["dataset_version"] = DATASET_VERSION

# Fill review-derived features for listings with no reviews before cutoff.
review_zero_fill_cols = [
    "total_reviews_before_cutoff",
    "unique_reviewers_before_cutoff",
    "avg_comment_len_before_cutoff",
    "max_comment_len_before_cutoff",
]

for col in review_zero_fill_cols:
    if col in feature_df.columns:
        feature_df[col] = feature_df[col].fillna(0)

if "total_reviews_before_cutoff" in feature_df.columns:
    feature_df["has_reviews_before_cutoff"] = (
        feature_df["total_reviews_before_cutoff"] > 0
    ).astype(int)

if "days_since_last_review" in feature_df.columns:
    # Sentinel for "no previous review".
    feature_df["days_since_last_review"] = feature_df["days_since_last_review"].fillna(PAST_WINDOW_DAYS + FUTURE_WINDOW_DAYS + 1)

# Fill calendar feature gaps conservatively.
calendar_count_cols = [
    col for col in feature_df.columns
    if col.startswith("calendar_days_observed") or col.startswith("available_days_last_")
]
for col in calendar_count_cols:
    feature_df[col] = feature_df[col].fillna(0)

assert feature_df["listing_id"].duplicated().sum() == 0

print(feature_df.shape)
display(feature_df.head())

(10480, 36)


,listing_id,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,...,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version,has_reviews_before_cutoff
0,27886,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,132.0,3,...,0.000000,3.0,30.0,30,0,0.0,1,2026-08-11,v1_student,1
1,28871,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,89.0,2,...,0.466667,2.0,730.0,30,21,0.7,0,2026-08-11,v1_student,1
2,29051,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,61.0,2,...,0.533333,2.0,730.0,30,0,0.0,1,2026-08-11,v1_student,1
3,44391,Entire home/apt,Entire rental unit,Centrum-Oost,4,2.0,NaN,1.5,NaN,3,...,0.000000,3.0,730.0,30,0,0.0,1,2026-08-11,v1_student,1
4,48373,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,4,2.0,NaN,1.5,NaN,3,...,0.000000,3.0,1125.0,30,0,0.0,1,2026-08-11,v1_student,1


## 14. Drop unusable columns

Before saving, remove bad feature columns.

Drop columns that are:

- more than 95% missing,
- constant across all rows,

but protect target/audit columns.

In [20]:
protected_columns = {
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
}

HIGH_MISSING_DROP_THRESHOLD = 0.95

# Compute missing rates.
missing_rates = feature_df.isna().mean()

# Drop columns with too much missingness, except protected columns.
high_missing_columns = [
    col for col, rate in missing_rates.items()
    if rate > HIGH_MISSING_DROP_THRESHOLD and col not in protected_columns
]

# Drop constant columns, except protected columns.
constant_columns = [
    col for col in feature_df.columns
    if col not in protected_columns and feature_df[col].nunique(dropna=False) <= 1
]

columns_to_drop = sorted(set(high_missing_columns + constant_columns))

print("High-missing columns:", high_missing_columns)
print("Constant columns:", constant_columns)
print("Columns to drop:", columns_to_drop)

feature_df = feature_df.drop(columns=columns_to_drop)

print("New shape:", feature_df.shape)
display(feature_df.head())

High-missing columns: []
Constant columns: ['calendar_days_observed_last_90d', 'calendar_days_observed_last_30d']
Columns to drop: ['calendar_days_observed_last_30d', 'calendar_days_observed_last_90d']
New shape: (10480, 34)


,listing_id,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,...,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version,has_reviews_before_cutoff
0,27886,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,132.0,3,...,0.000000,3.0,30.0,30,0,0.0,1,2026-08-11,v1_student,1
1,28871,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,89.0,2,...,0.466667,2.0,730.0,30,21,0.7,0,2026-08-11,v1_student,1
2,29051,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,61.0,2,...,0.533333,2.0,730.0,30,0,0.0,1,2026-08-11,v1_student,1
3,44391,Entire home/apt,Entire rental unit,Centrum-Oost,4,2.0,NaN,1.5,NaN,3,...,0.000000,3.0,730.0,30,0,0.0,1,2026-08-11,v1_student,1
4,48373,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,4,2.0,NaN,1.5,NaN,3,...,0.000000,3.0,1125.0,30,0,0.0,1,2026-08-11,v1_student,1


## 15. Validate the final dataset

The validation step is mandatory.

Check:

1. no duplicate `listing_id + cutoff_date`,
2. target exists and is binary,
3. no missing target values,
4. no forbidden PII columns,
5. no future leakage columns in model inputs.

In [21]:
duplicate_count = feature_df.duplicated(subset=["listing_id", "cutoff_date"]).sum()
missing_target_count = feature_df["high_demand_proxy"].isna().sum()
unique_target_values = sorted(feature_df["high_demand_proxy"].dropna().astype(int).unique().tolist())

forbidden_columns = {
    "host_id",
    "host_pseudo_id",
    "reviewer_id",
    "reviewer_pseudo_id",
    "review_id",
    "license",
    "bathrooms_text",
}

present_forbidden_columns = sorted(forbidden_columns.intersection(feature_df.columns))

label_only_columns = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

model_input_columns = [
    col for col in feature_df.columns
    if col not in label_only_columns
    and col not in ["listing_id", "cutoff_date", "dataset_version"]
]

future_leakage_columns = [
    col for col in model_input_columns
    if col.startswith("future_")
]

# Required validation rules.
assert duplicate_count == 0, "Final dataset has duplicate listing_id + cutoff_date rows."
assert missing_target_count == 0, "Target column has missing values."
assert set(unique_target_values).issubset({0, 1}), "Target must be binary 0/1."
assert len(unique_target_values) >= 1, "Target should contain at least one observed class."
assert len(present_forbidden_columns) == 0, f"Forbidden PII columns are present: {present_forbidden_columns}"
assert len(future_leakage_columns) == 0, f"Future leakage columns found in model inputs: {future_leakage_columns}"
assert len(model_input_columns) > 0, "No model input columns found."

print("duplicate_count:", duplicate_count)
print("missing_target_count:", missing_target_count)
print("unique_target_values:", unique_target_values)
print("present_forbidden_columns:", present_forbidden_columns)
print("future_leakage_columns:", future_leakage_columns)
print("model_input_column_count:", len(model_input_columns))

duplicate_count: 0
missing_target_count: 0
unique_target_values: [0, 1]
present_forbidden_columns: []
future_leakage_columns: []
model_input_column_count: 27


In [22]:
missing_report = (
    feature_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_report.columns = ["column", "missing_rate"]

calendar_coverage_summary_df = (
    feature_df[["future_calendar_days_observed_30d"]]
    .describe()
    .reset_index()
)

display(missing_report.head(30))
display(label_distribution)
display(calendar_coverage_summary_df)

,column,missing_rate
0,listing_price,0.439504
1,beds,0.436641
2,bedrooms,0.029198
3,is_superhost,0.010973
4,bathrooms,0.001145
5,listing_id,0.000000
6,room_type,0.000000
7,accommodates,0.000000
8,property_type,0.000000
9,neighbourhood_name,0.000000


,high_demand_proxy,count,percentage
0,1,7994,0.7628
1,0,2486,0.2372


,index,future_calendar_days_observed_30d
0,count,10480.0
1,mean,30.0
2,std,0.0
3,min,30.0
4,25%,30.0
5,50%,30.0
6,75%,30.0
7,max,30.0


## 16. Save versioned outputs

Save:

- feature dataset,
- metadata,
- validation report,
- PII audit.

The MLflow notebook must read this output instead of querying raw database tables again.

In [23]:
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"
validation_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_validation_report.json"
pii_audit_path = FEATURE_DIR / f"pii_audit_{DATASET_VERSION}.csv"

feature_df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

try:
    feature_df.to_parquet(parquet_path, index=False)
    print("Saved Parquet:", parquet_path)
except ImportError:
    print("Parquet not saved because pyarrow/fastparquet is not installed.")
    print("Install pyarrow with: pip install pyarrow")

def json_safe(value):
    """Convert numpy/pandas/date objects to plain JSON-serializable Python values."""
    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass

    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if hasattr(value, "isoformat"):
        return value.isoformat()

    return value

calendar_coverage_columns = [
    col for col in feature_df.columns
    if col.startswith("calendar_days_observed")
    or col.startswith("future_calendar_days_observed")
]

calendar_coverage_summary = {
    col: {
        "min": json_safe(feature_df[col].min()),
        "median": json_safe(feature_df[col].median()),
        "max": json_safe(feature_df[col].max()),
        "missing_rate": json_safe(feature_df[col].isna().mean()),
    }
    for col in calendar_coverage_columns
}

metadata = {
    "dataset_version": DATASET_VERSION,
    "entity_column": ENTITY_COLUMN,
    "cutoff_date": str(cutoff_date),
    "history_start_date": str(history_start_date),
    "label_end_date": str(label_end_date),
    "past_window_days": PAST_WINDOW_DAYS,
    "future_window_days": FUTURE_WINDOW_DAYS,
    "source_tables": [
        "core.listing",
        "core.host",
        "core.neighbourhood",
        "core.review",
        "core.calendar_day",
    ],
    "target_definition": {
        "label_column": "high_demand_proxy",
        "description": "1 when future_available_rate_30d <= threshold; otherwise 0",
        "threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
        "warning": "This is a low-availability proxy, not confirmed booking demand.",
    },
    "privacy_rules": {
        "listing_id": "kept as entity key only; exclude from model inputs",
        "forbidden_raw_columns": sorted(forbidden_columns),
        "raw_review_text": "not present in this DB; only numeric comment_len aggregates are used",
    },
    "feature_columns": model_input_columns,
    "label_columns": label_only_columns,
    "dropped_columns": columns_to_drop,
    "shape": {
        "rows": int(feature_df.shape[0]),
        "columns": int(feature_df.shape[1]),
    },
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

validation_report = {
    "duplicate_listing_cutoff_rows": int(duplicate_count),
    "missing_target_count": int(missing_target_count),
    "target_values": [int(x) for x in unique_target_values],
    "present_forbidden_columns": present_forbidden_columns,
    "future_leakage_columns_in_model_inputs": future_leakage_columns,
    "model_input_column_count": int(len(model_input_columns)),
    "missing_report": missing_report.to_dict(orient="records"),
    "label_distribution": label_distribution.to_dict(orient="records"),
    "calendar_coverage_summary": calendar_coverage_summary,
}

with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)

pii_audit.to_csv(pii_audit_path, index=False)

print("Saved metadata:", metadata_path)
print("Saved validation report:", validation_path)
print("Saved PII audit:", pii_audit_path)

Saved CSV: /home/alireza/PycharmProjects/Quera-Homeworks/HW02-A/data/features/listing_availability_features_v1_student.csv
Saved Parquet: /home/alireza/PycharmProjects/Quera-Homeworks/HW02-A/data/features/listing_availability_features_v1_student.parquet
Saved metadata: /home/alireza/PycharmProjects/Quera-Homeworks/HW02-A/data/features/listing_availability_features_v1_student_metadata.json
Saved validation report: /home/alireza/PycharmProjects/Quera-Homeworks/HW02-A/data/features/listing_availability_features_v1_student_validation_report.json
Saved PII audit: /home/alireza/PycharmProjects/Quera-Homeworks/HW02-A/data/features/pii_audit_v1_student.csv


## 17. Final preview

Use this final cell to confirm the output shape and columns.

Before moving to Notebook 2, make sure:

- target column exists,
- model input columns do not include future columns,
- no forbidden PII columns are present,
- saved files exist in `data/features/`.

In [24]:
print("Final shape:", feature_df.shape)

display(feature_df.head())

feature_df.info()

Final shape: (10480, 34)


,listing_id,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,...,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version,has_reviews_before_cutoff
0,27886,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,132.0,3,...,0.000000,3.0,30.0,30,0,0.0,1,2026-08-11,v1_student,1
1,28871,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,89.0,2,...,0.466667,2.0,730.0,30,21,0.7,0,2026-08-11,v1_student,1
2,29051,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,61.0,2,...,0.533333,2.0,730.0,30,0,0.0,1,2026-08-11,v1_student,1
3,44391,Entire home/apt,Entire rental unit,Centrum-Oost,4,2.0,NaN,1.5,NaN,3,...,0.000000,3.0,730.0,30,0,0.0,1,2026-08-11,v1_student,1
4,48373,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,4,2.0,NaN,1.5,NaN,3,...,0.000000,3.0,1125.0,30,0,0.0,1,2026-08-11,v1_student,1


<class 'pandas.DataFrame'>
RangeIndex: 10480 entries, 0 to 10479
Data columns (total 34 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   listing_id                            10480 non-null  int64  
 1   room_type                             10480 non-null  str    
 2   property_type                         10480 non-null  str    
 3   neighbourhood_name                    10480 non-null  str    
 4   accommodates                          10480 non-null  int64  
 5   bedrooms                              10174 non-null  float64
 6   beds                                  5904 non-null   float64
 7   bathrooms                             10468 non-null  float64
 8   listing_price                         5874 non-null   float64
 9   minimum_nights                        10480 non-null  int64  
 10  maximum_nights                        10480 non-null  int64  
 11  instant_bookable          